In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [4]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="speech-uk/voice-of-america", 
                  repo_type="dataset", local_dir="./voice-of-america")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 85 files: 100%|██████████| 85/85 [00:26<00:00,  3.16it/s]


'/home/ubuntu/voice-of-america'

In [7]:
files = glob('voice-of-america/*/*.parquet')
len(files)

83

In [9]:
df = pd.read_parquet(files[0])
df

,audio,duration,transcription
0,{'bytes': b'RIFF2W\x06\x00WAVEfmt \x12\x00\x00...,6.492,російський лідер говорив що підтримує діалогу ...
1,{'bytes': b'RIFF2g\x02\x00WAVEfmt \x12\x00\x00...,2.460,що не визнає білоруського позицію
2,{'bytes': b'RIFF2o\x05\x00WAVEfmt \x12\x00\x00...,5.564,хоча такі лідери як світлана тихановська нагол...
3,{'bytes': b'RIFF2W\x07\x00WAVEfmt \x12\x00\x00...,7.516,якщо хтось з тих опозиційних лідерів які ви б ...
4,{'bytes': b'RIFF2\x9f\x00\x00WAVEfmt \x12\x00\...,0.636,такого
...,...,...,...
3925,{'bytes': b'RIFF2\xa7\x10\x00WAVEfmt \x12\x00\...,17.052,і того щоб з ними боротися ми все ж таки розра...
3926,{'bytes': b'RIFF2\x0f\x06\x00WAVEfmt \x12\x00\...,6.204,надалі такі дії сполучених штатів америки вони...
3927,{'bytes': b'RIFF2g\x08\x00WAVEfmt \x12\x00\x00...,8.604,і для нас це буде також певним кроком для боль...
3928,{'bytes': b'RIFF2\x7f\t\x00WAVEfmt \x12\x00\x0...,9.724,олено ви говорите про реформи сьогодні але баг...


In [14]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [15]:
# data = loop((files[:1], 0))
# data

In [16]:
len(files)

83

In [18]:
# data = multiprocessing(files, loop, cores = 40)

In [19]:
len(data)

313751

In [21]:
with open('voice-of-america.json', 'w') as fopen:
    json.dump(data, fopen)

In [22]:
audio_files = [d['audio_filename'] for d in data]

with open('voice-of-america-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [23]:
len(list(set(audio_files)))

313751